# Convert the trained LoRA adapter to a single GGUF file

This is a follow-up to the training notebook, for when Ollama's `ADAPTER <folder>` (safetensors
directory) fails with `Error: no Modelfile or safetensors files found` — a known bug in Ollama on
Windows. A single-file `ADAPTER ./file.gguf` reference avoids it.

**No GPU needed for this notebook** — it's pure CPU tensor conversion, so there's no quota to worry
about. Runtime → Change runtime type → CPU is fine (or leave as-is).

**Upload these files when prompted** (the ones you already downloaded from the training run):
`adapter_config.json`, `adapter_model.safetensors`, `tokenizer.json`, `tokenizer_config.json`,
`special_tokens_map.json`, `chat_template.jinja`


## 1. Upload the adapter files

In [ ]:
import os
os.makedirs("adapter", exist_ok=True)

from google.colab import files
uploaded = files.upload()
import shutil
for fname in uploaded:
    shutil.move(fname, os.path.join("adapter", fname))

required = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in required if not os.path.exists(os.path.join("adapter", f))]
assert not missing, f"Missing required files: {missing}"
print("Got:", os.listdir("adapter"))


## 2. Get the conversion script (from llama.cpp) and install dependencies

In [ ]:
!git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
!pip install -q gguf safetensors huggingface_hub


## 3. Convert

`--base-model-id` fetches just the small base model config (architecture/hyperparameters, not the
full weights) from Hugging Face, to know the tensor shapes — matches the model we trained on.


In [ ]:
!python llama.cpp/convert_lora_to_gguf.py adapter \
    --outfile edhkl_clinical_notes.gguf \
    --outtype f16 \
    --base-model-id unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit

!ls -la edhkl_clinical_notes.gguf


## 4. Download the result

In [ ]:
from google.colab import files
files.download("edhkl_clinical_notes.gguf")


## 5. Use it in Ollama

1. Put `edhkl_clinical_notes.gguf` inside your `V2textDoc` project, e.g.
   `V2textDoc\models\edhkl_clinical_notes.gguf` (a single file this time, not a folder).
2. In `V2textDoc\models\`, edit (or recreate) `Modelfile` to contain exactly:
   ```
   FROM llama3.1:8b
   ADAPTER ./edhkl_clinical_notes.gguf
   ```
3. From a terminal in `V2textDoc\models\`:
   ```
   ollama create edhkl-clinical-notes -f Modelfile
   ```
4. In `backend\.env`, set `OLLAMA_MODEL=edhkl-clinical-notes`, then
   `docker compose down && docker compose up` (no rebuild needed).
